# Phase 3 — Final Test Evaluation & E-Improvement Analysis

Compares three approaches on the held-out test set (n=288):
- **v1**: Original winner (`trait_first`, 11-shot, temp 0.3, original exemplars)
- **v2**: E-improved (`trait_first_v2`, rebalanced exemplars, E-calibration guards)
- **v1+cal**: v1 predictions with post-hoc linear E calibration (fitted on dev)

In [ ]:
import json
import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 12, "axes.labelsize": 10})

TRAITS = ["O", "C", "E", "A", "N"]
TRAIT_NAMES = {"O": "Openness", "C": "Conscientiousness", "E": "Extraversion", "A": "Agreeableness", "N": "Neuroticism"}
BASELINE_R = 0.457
RESULTS = Path("results")

In [ ]:
# Load results
def load_results(pattern):
    rows = []
    for fp in sorted(RESULTS.glob(pattern)):
        for line in open(fp):
            r = json.loads(line.strip())
            if r.get("detected_ocean"):
                rows.append(r)
    # deduplicate by sample_id (keep last)
    seen = {}
    for r in rows:
        seen[r["sample_id"]] = r
    return list(seen.values())

v1_test = load_results("phase3_v1_test_*.jsonl")
v2_test = load_results("phase3_v2_test_*.jsonl")
v2_dev = load_results("phase3_v2_dev_*.jsonl")

# Also load v1 dev predictions for calibration fitting
import glob as g
v1_dev_rows = []
for fp in sorted(g.glob(str(Path("..") / "results" / "harness_trait_first_s11_t03*.jsonl"))):
    for line in open(fp):
        r = json.loads(line.strip())
        if r.get("detected_ocean"):
            v1_dev_rows.append(r)
v1_dev_seen = {}
for r in v1_dev_rows:
    v1_dev_seen[r.get("sample_id")] = r
v1_dev = list(v1_dev_seen.values())

print(f"v1 test: {len(v1_test)}, v2 test: {len(v2_test)}, v2 dev: {len(v2_dev)}, v1 dev: {len(v1_dev)}")

In [ ]:
# Fit E calibration on v1 dev predictions
v1_dev_e_pred = np.array([r["detected_ocean"]["E"] for r in v1_dev if "E" in r.get("ground_truth_ocean", {})])
v1_dev_e_gt = np.array([r["ground_truth_ocean"]["E"] for r in v1_dev if "E" in r.get("ground_truth_ocean", {})])

slope, intercept, _, _, _ = stats.linregress(v1_dev_e_pred, v1_dev_e_gt)
print(f"E calibration: calibrated_E = {slope:.4f} * pred_E + ({intercept:+.4f})")
print(f"Dev fit — before: r={stats.pearsonr(v1_dev_e_pred, v1_dev_e_gt)[0]:.3f}, bias={np.mean(v1_dev_e_pred - v1_dev_e_gt):+.3f}")
cal_dev = np.clip(slope * v1_dev_e_pred + intercept, -1, 1)
print(f"Dev fit — after:  r={stats.pearsonr(cal_dev, v1_dev_e_gt)[0]:.3f}, bias={np.mean(cal_dev - v1_dev_e_gt):+.3f}")

In [ ]:
# Compute metrics for all three approaches on test set
def compute_metrics(rows, e_calibration=None):
    results = {}
    for t in TRAITS:
        pairs = [(r["detected_ocean"][t], r["ground_truth_ocean"][t])
                 for r in rows if t in r.get("ground_truth_ocean", {}) and t in r.get("detected_ocean", {})]
        if len(pairs) < 5:
            continue
        pred = np.array([p[0] for p in pairs])
        gt = np.array([p[1] for p in pairs])
        if t == "E" and e_calibration is not None:
            pred = np.clip(e_calibration[0] * pred + e_calibration[1], -1, 1)
        r_val, p_val = stats.pearsonr(pred, gt)
        results[t] = {
            "r": r_val, "mae": np.mean(np.abs(pred - gt)),
            "bias": np.mean(pred - gt), "n": len(pairs),
            "pred": pred, "gt": gt,
        }
    results["macro_r"] = np.mean([results[t]["r"] for t in TRAITS if t in results])
    results["macro_mae"] = np.mean([results[t]["mae"] for t in TRAITS if t in results])
    return results

m_v1 = compute_metrics(v1_test)
m_v2 = compute_metrics(v2_test)
m_v1cal = compute_metrics(v1_test, e_calibration=(slope, intercept))

# Summary table
rows_t = []
for label, m in [("v1 (original)", m_v1), ("v2 (E-improved)", m_v2), ("v1+cal (calibrated)", m_v1cal)]:
    row = {"Approach": label}
    for t in TRAITS:
        if t in m:
            row[f"r_{t}"] = f"{m[t]['r']:.3f}"
            row[f"MAE_{t}"] = f"{m[t]['mae']:.3f}"
            row[f"bias_{t}"] = f"{m[t]['bias']:+.3f}"
    row["macro_r"] = f"{m['macro_r']:.3f}"
    row["macro_MAE"] = f"{m['macro_mae']:.3f}"
    rows_t.append(row)

summary_df = pd.DataFrame(rows_t).set_index("Approach")
display(summary_df)

In [ ]:
# Grouped bar chart: Pearson r per trait for all 3 approaches
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Panel A: Pearson r
ax = axes[0]
x = np.arange(5)
w = 0.25
for i, (label, m, color) in enumerate([
    ("v1 original", m_v1, "steelblue"),
    ("v2 E-improved", m_v2, "coral"),
    ("v1+calibration", m_v1cal, "mediumpurple")]):
    vals = [m[t]["r"] if t in m else 0 for t in TRAITS]
    bars = ax.bar(x + i*w - w, vals, w, label=label, color=color, edgecolor="white")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.2f}", ha="center", fontsize=7, rotation=90, va="bottom")

ax.axhline(BASELINE_R, color="gray", ls="--", lw=1, label=f"baseline r={BASELINE_R}")
ax.set_xticks(x)
ax.set_xticklabels([TRAIT_NAMES[t] for t in TRAITS], fontsize=9)
ax.set_ylabel("Pearson r")
ax.set_title("A) Per-Trait Correlation (Test Set)")
ax.legend(fontsize=7, loc="lower left")
ax.set_ylim(0, 1)

# Panel B: Bias
ax = axes[1]
for i, (label, m, color) in enumerate([
    ("v1 original", m_v1, "steelblue"),
    ("v2 E-improved", m_v2, "coral"),
    ("v1+calibration", m_v1cal, "mediumpurple")]):
    vals = [m[t]["bias"] if t in m else 0 for t in TRAITS]
    bars = ax.bar(x + i*w - w, vals, w, label=label, color=color, edgecolor="white")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01 * np.sign(v), f"{v:+.2f}", ha="center", fontsize=7,
                rotation=90, va="bottom" if v >= 0 else "top")

ax.axhline(0, color="gray", ls="--", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels([TRAIT_NAMES[t] for t in TRAITS], fontsize=9)
ax.set_ylabel("Mean bias (pred - gt)")
ax.set_title("B) Per-Trait Prediction Bias (Test Set)")
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# E-specific deep dive: bias by GT bin for all 3 approaches
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
bins_edges = [(-1.0, -0.5), (-0.5, 0.0), (0.0, 0.5), (0.5, 1.0)]
bin_labels = ["[-1,-.5)", "[-.5, 0)", "[0, .5)", "[.5, 1]"]

for ax, (label, m, color) in zip(axes, [
    ("v1 original", m_v1, "steelblue"),
    ("v2 E-improved", m_v2, "coral"),
    ("v1+calibration", m_v1cal, "mediumpurple")]):
    if "E" not in m:
        continue
    pred, gt = m["E"]["pred"], m["E"]["gt"]
    biases, counts = [], []
    for lo, hi in bins_edges:
        mask = (gt >= lo) & (gt < hi + 0.01)
        if mask.sum() >= 2:
            biases.append((pred[mask] - gt[mask]).mean())
            counts.append(mask.sum())
        else:
            biases.append(0)
            counts.append(0)

    bar_colors = ["#d32f2f" if b > 0.3 else "#f57c00" if b > 0.1 else "#4caf50" if abs(b) <= 0.1 else "#1976d2" for b in biases]
    bars = ax.bar(range(4), biases, color=bar_colors, edgecolor="white", width=0.7)
    ax.set_xticks(range(4))
    ax.set_xticklabels(bin_labels, fontsize=8)
    ax.axhline(0, color="gray", ls="--", lw=0.8)
    ax.set_ylim(-0.5, 1.2)
    ax.set_title(f"{label}\nE: r={m['E']['r']:.3f}, bias={m['E']['bias']:+.3f}", fontsize=10)
    ax.set_ylabel("Mean bias")
    for i, (b, n) in enumerate(zip(biases, counts)):
        ax.text(i, b + 0.04 * np.sign(b), f"n={n}\n{b:+.2f}", ha="center", fontsize=8,
                va="bottom" if b >= 0 else "top")

fig.suptitle("Extraversion Prediction Bias by GT Bin (Test Set)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: pred vs GT for E across all 3 approaches
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (label, m, color) in zip(axes, [
    ("v1 original", m_v1, "steelblue"),
    ("v2 E-improved", m_v2, "coral"),
    ("v1+calibration", m_v1cal, "mediumpurple")]):
    if "E" not in m:
        continue
    pred, gt = m["E"]["pred"], m["E"]["gt"]
    ax.scatter(gt, pred, s=15, alpha=0.5, color=color, edgecolors="none")
    ax.plot([-1, 1], [-1, 1], "k--", lw=0.8, alpha=0.5)
    ax.set_xlabel("GT Extraversion")
    ax.set_ylabel("Predicted Extraversion")
    ax.set_title(f"{label}\nr={m['E']['r']:.3f}  MAE={m['E']['mae']:.3f}  bias={m['E']['bias']:+.3f}", fontsize=10)
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect("equal")
    ax.grid(alpha=0.15)

fig.suptitle("Extraversion: Predicted vs Ground Truth (Test Set)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Full 5-trait scatter for the best approach (v1)
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
best = m_v1

for ax, t in zip(axes, TRAITS):
    if t not in best:
        continue
    pred, gt = best[t]["pred"], best[t]["gt"]
    ax.scatter(gt, pred, s=12, alpha=0.4, color="steelblue", edgecolors="none")
    ax.plot([-1, 1], [-1, 1], "k--", lw=0.8, alpha=0.5)
    ax.set_xlabel("Ground Truth")
    if t == "O":
        ax.set_ylabel("Predicted")
    ax.set_title(f"{TRAIT_NAMES[t]}\nr={best[t]['r']:.3f}  n={best[t]['n']}", fontsize=10)
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect("equal")
    ax.grid(alpha=0.15)

fig.suptitle("v1 Original — All Traits: Predicted vs Ground Truth (Test Set)", fontsize=13, y=1.04)
plt.tight_layout()
plt.show()

In [ ]:
# Bootstrap 95% CIs for macro r
def bootstrap_macro_r(rows, n_boot=2000, e_cal=None, seed=42):
    rng = np.random.RandomState(seed)
    macro_rs = []
    for _ in range(n_boot):
        idx = rng.choice(len(rows), len(rows), replace=True)
        sampled = [rows[i] for i in idx]
        trait_rs = []
        for t in TRAITS:
            pairs = [(r["detected_ocean"][t], r["ground_truth_ocean"][t])
                     for r in sampled if t in r.get("ground_truth_ocean", {}) and t in r.get("detected_ocean", {})]
            if len(pairs) < 5:
                continue
            p = np.array([x[0] for x in pairs])
            g = np.array([x[1] for x in pairs])
            if t == "E" and e_cal is not None:
                p = np.clip(e_cal[0] * p + e_cal[1], -1, 1)
            if np.std(p) < 1e-9 or np.std(g) < 1e-9:
                continue
            r_val, _ = stats.pearsonr(p, g)
            trait_rs.append(r_val)
        if trait_rs:
            macro_rs.append(np.mean(trait_rs))
    return np.array(macro_rs)

boot_v1 = bootstrap_macro_r(v1_test)
boot_v2 = bootstrap_macro_r(v2_test)
boot_cal = bootstrap_macro_r(v1_test, e_cal=(slope, intercept))

fig, ax = plt.subplots(figsize=(10, 4))
for i, (label, boot, color) in enumerate([
    ("v1 original", boot_v1, "steelblue"),
    ("v2 E-improved", boot_v2, "coral"),
    ("v1+calibration", boot_cal, "mediumpurple")]):
    ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
    mean = boot.mean()
    ax.barh(i, mean, color=color, edgecolor="white", height=0.5)
    ax.plot([ci_lo, ci_hi], [i, i], color="black", lw=2)
    ax.text(ci_hi + 0.005, i, f"{mean:.3f} [{ci_lo:.3f}, {ci_hi:.3f}]", va="center", fontsize=9)

ax.axvline(BASELINE_R, color="gray", ls="--", lw=1, label=f"baseline={BASELINE_R}")
ax.set_yticks(range(3))
ax.set_yticklabels(["v1 original", "v2 E-improved", "v1+calibration"])
ax.set_xlabel("Macro Pearson r")
ax.set_title("Test Set: Macro r with Bootstrap 95% CI")
ax.legend(fontsize=8)
ax.set_xlim(0.3, 0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Final summary
print("=" * 70)
print("  PHASE 3 — FINAL TEST SET RESULTS")
print("=" * 70)
print(f"  Baseline (N8N, 7-shot):       macro r = {BASELINE_R:.3f}")
print(f"  v1 (trait_first, 11-shot):     macro r = {m_v1['macro_r']:.3f}  macro MAE = {m_v1['macro_mae']:.3f}")
print(f"  v2 (E-improved prompt):        macro r = {m_v2['macro_r']:.3f}  macro MAE = {m_v2['macro_mae']:.3f}")
print(f"  v1+cal (post-hoc E calibr.):   macro r = {m_v1cal['macro_r']:.3f}  macro MAE = {m_v1cal['macro_mae']:.3f}")
print()
print("  Per-trait Pearson r (test):")
for t in TRAITS:
    v1r = m_v1[t]['r'] if t in m_v1 else float('nan')
    v2r = m_v2[t]['r'] if t in m_v2 else float('nan')
    vcr = m_v1cal[t]['r'] if t in m_v1cal else float('nan')
    best = max(v1r, v2r, vcr)
    marker = {v1r: 'v1', v2r: 'v2', vcr: 'cal'}[best]
    print(f"    {TRAIT_NAMES[t]:<20s}  v1={v1r:.3f}  v2={v2r:.3f}  cal={vcr:.3f}  best={marker}")
print()
print(f"  Improvement over baseline:")
print(f"    v1:   +{m_v1['macro_r']-BASELINE_R:.3f} ({(m_v1['macro_r']-BASELINE_R)/BASELINE_R*100:+.0f}%)")
print(f"    v2:   +{m_v2['macro_r']-BASELINE_R:.3f} ({(m_v2['macro_r']-BASELINE_R)/BASELINE_R*100:+.0f}%)")
print(f"    cal:  +{m_v1cal['macro_r']-BASELINE_R:.3f} ({(m_v1cal['macro_r']-BASELINE_R)/BASELINE_R*100:+.0f}%)")
print("=" * 70)